<a href="https://colab.research.google.com/github/yamms2340/researchWorkCodes/blob/main/CnnModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn

class Cnn1d(nn.Module):
    def __init__(self):
        super().__init__()
        # Input shape: (Batch Size, 1 Channel, 1000 Time Steps)
        self.c1 = nn.Conv1d(in_channels=1, out_channels=16, kernel_size=5)

        # Dilation=2 expands the receptive field to look past immediate adjacent noise
        self.c2 = nn.Conv1d(in_channels=16, out_channels=32, kernel_size=3, dilation=2)

        self.r = nn.ReLU()

        # Flattened size: 32 channels * 992 remaining spatial dimensions
        self.fc = nn.Linear(32 * 992, 3)

    def forward(self, x):
        x = self.r(self.c1(x))
        x = self.r(self.c2(x))

        # Flatten the tensor for the fully connected layer
        x = x.view(x.shape[0], -1)

        return self.fc(x)

In [ ]:
import pandas as pd
import torch
import numpy as np

# (Assuming your Cnn1d class is defined above this)

# 1. Load your dataset
df = pd.read_csv("lorenz_data.csv")

# 2. Extract and normalize your 1000-point z-variable
z_data = df['z_var'].values
# Normalization is crucial so the CNN doesn't get overwhelmed!
z_norm = (z_data - np.min(z_data)) / (np.max(z_data) - np.min(z_data) + 1e-8)
# Convert to PyTorch Tensor
tensor_data = torch.tensor(z_norm, dtype=torch.float32).view(1, 1, -1)

# 3. Build the "brain" using the blueprint
model = Cnn1d()

# 4. Hand the data to the brain to get your 3 exponents!
# We use .detach().numpy()[0] to clean the PyTorch tensor into a standard array
model.eval() # Tell the model we are testing, not training
with torch.no_grad():
    predicted_exponents = model(tensor_data).numpy()[0]

print("\n--- CNN Predicted Spectrum ---")
print(f"LE1 (Maximal):     {predicted_exponents[0]:.4f}")
print(f"LE2 (Marginal):    {predicted_exponents[1]:.4f}")
print(f"LE3 (Contractive): {predicted_exponents[2]:.4f}")

# 5. Save CNN Results to a separate file
df_cnn = pd.DataFrame({
    "LE_Type": ["LE1 (Max)", "LE2 (Marg)", "LE3 (Cont)"],
    "CNN_Prediction": predicted_exponents  # <--- Notice we use the variable from Step 4 here
})
df_cnn.to_csv("cnn_results.csv", index=False)

print("Successfully saved to 'cnn_results.csv'")


--- CNN Predicted Spectrum ---
LE1 (Maximal):     -0.0367
LE2 (Marginal):    -0.0169
LE3 (Contractive): -0.0588
Successfully saved to 'cnn_results.csv'
